## Занятие 6. Группировки и сводные таблицы (Group by)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('sales_full.csv')
df['Date'] = pd.to_datetime(df['Date'])

### Задание 1. Базовые агрегации по региону

In [2]:
df.head()

,Date,Region,Manager,Category,Revenue,Quantity,Discount
0,2026-01-14,East,Frank,Books,432.55,1,0.17
1,2026-01-04,West,Jack,Home,421.92,7,0.04
2,2026-02-23,West,Irene,Toys,1928.94,14,0.14
3,2026-01-10,West,Henry,Home,571.02,7,0.19
4,2026-01-14,East,Grace,Home,671.76,9,0.18


In [3]:
region_stats = df.groupby('Region')['Revenue'].agg(['sum', 'mean', 'count', 'std'])
region_stats = region_stats.sort_values('sum', ascending=False)
region_stats

,sum,mean,count,std
Region,,,,
West,64742.49,1061.352295,61,553.567423
South,60839.34,965.703810,63,526.967434
East,48216.04,1004.500833,48,523.508922
North,29054.36,1037.655714,28,551.419105


### Задание 2. Группировка по двум признакам (Регион + Категория)

In [4]:
region_category_mean = df.groupby(['Region', 'Category'])['Revenue'].mean().unstack()
region_category_mean

Category,Books,Clothing,Electronics,Home,Toys
Region,,,,,
East,899.250000,814.251667,1003.722000,1001.131818,1258.531250
North,1098.920000,1282.948333,1070.877273,796.626667,866.473333
South,1259.593333,894.475000,869.315000,989.310000,980.870833
West,839.301111,976.855000,1273.494545,968.382000,1448.104286


In [5]:
pivot_margins = pd.pivot_table(
    df,
    values='Revenue',
    index='Region',
    columns='Category',
    aggfunc='mean',
    margins=True
)
pivot_margins

Category,Books,Clothing,Electronics,Home,Toys,All
Region,,,,,,
East,899.250000,814.251667,1003.722000,1001.131818,1258.531250,1004.500833
North,1098.920000,1282.948333,1070.877273,796.626667,866.473333,1037.655714
South,1259.593333,894.475000,869.315000,989.310000,980.870833,965.703810
West,839.301111,976.855000,1273.494545,968.382000,1448.104286,1061.352295
All,980.124400,969.545500,1039.496667,962.557778,1152.495000,1014.261150


### Задание 3. Множественные агрегации для разных полей

In [6]:
manager_stats = df.groupby('Manager').agg({
    'Revenue': ['count', 'sum', 'mean'],
    'Discount': 'mean'
})
manager_stats.columns = ['Транзакций', 'Общая выручка', 'Средняя выручка', 'Средняя скидка']
manager_stats = manager_stats.sort_values('Общая выручка', ascending=False)
manager_stats

,Транзакций,Общая выручка,Средняя выручка,Средняя скидка
Manager,,,,
Jack,24,27057.21,1127.383750,0.143750
Grace,23,24858.45,1080.802174,0.168696
Frank,25,23357.59,934.303600,0.132400
Charlie,21,23257.03,1107.477619,0.126190
Henry,20,21353.67,1067.683500,0.138000
Diana,23,19733.78,857.990435,0.166087
Anna,19,19377.47,1019.866842,0.163684
Eve,19,17848.53,939.396316,0.154211
Irene,17,16331.61,960.682941,0.173529


### Задание 4. Фильтрация групп и трансформация

In [7]:
region_revenue = df.groupby('Region')['Revenue'].sum()
print('Общая выручка по регионам:')
print(region_revenue)

rich_regions = df.groupby('Region').filter(lambda x: x['Revenue'].sum() > 15000)
print()
print('Регионы с общей выручкой > 15000:', sorted(rich_regions['Region'].unique()))
print(f"Осталось строк: {len(rich_regions)} из {len(df)}")

Общая выручка по регионам:
Region
East     48216.04
North    29054.36
South    60839.34
West     64742.49
Name: Revenue, dtype: float64

Регионы с общей выручкой > 15000: ['East', 'North', 'South', 'West']
Осталось строк: 200 из 200


In [8]:
df['Revenue_WithinRegion_Norm'] = df.groupby('Region')['Revenue'].transform(
    lambda x: (x - x.mean()) / x.std()
)

df[['Date', 'Region', 'Revenue', 'Revenue_WithinRegion_Norm']].head(10)

,Date,Region,Revenue,Revenue_WithinRegion_Norm
0,2026-01-14,East,432.55,-1.092533
1,2026-01-04,West,421.92,-1.155112
2,2026-02-23,West,1928.94,1.567267
3,2026-01-10,West,571.02,-0.885768
4,2026-01-14,East,671.76,-0.635597
5,2026-03-24,East,1233.40,0.437240
6,2026-03-09,South,1218.49,0.479700
7,2026-03-26,North,1539.10,0.909371
8,2026-02-04,South,952.12,-0.025777
9,2026-02-15,South,240.36,-1.376449


In [9]:
check = df.groupby('Region')['Revenue_WithinRegion_Norm'].agg(['mean', 'std'])
check

,mean,std
Region,,
East,-1.035052e-16,1.0
North,-9.516197e-17,1.0
South,-1.057355e-16,1.0
West,4.108735e-16,1.0


### Задание 5. Сводная таблица с несколькими значениями и группировкой по дате

In [10]:
df['Month'] = df['Date'].dt.month
df[['Date', 'Month']].head()

,Date,Month
0,2026-01-14,1
1,2026-01-04,1
2,2026-02-23,2
3,2026-01-10,1
4,2026-01-14,1


In [11]:
pivot_month = pd.pivot_table(
    df,
    values='Revenue',
    index='Month',
    columns='Category',
    aggfunc=['sum', 'mean']
)
pivot_month

sum                                                   mean  \
Category     Books  Clothing Electronics      Home      Toys        Books   
Month                                                                       
1          5605.48   9004.54    20561.70  13127.60   9379.13   700.685000   
2          7636.27  14543.02    16752.51  18629.69   7895.88  1272.711667   
3         11261.36  15234.26    15700.12  20220.83  17299.84  1023.760000   

                                                              
Category     Clothing  Electronics         Home         Toys  
Month                                                         
1          900.454000  1142.316667   772.211765  1172.391250  
2          969.534667   985.441765  1164.355625  1127.982857  
3         1015.617333   981.257500   962.896667  1153.322667

In [12]:
pivot_month_sum = pd.pivot_table(
    df,
    values='Revenue',
    index='Month',
    columns='Category',
    aggfunc='sum'
)
pivot_month_sum

Category,Books,Clothing,Electronics,Home,Toys
Month,,,,,
1,5605.48,9004.54,20561.70,13127.60,9379.13
2,7636.27,14543.02,16752.51,18629.69,7895.88
3,11261.36,15234.26,15700.12,20220.83,17299.84


In [13]:
pivot_month_mean = pd.pivot_table(
    df,
    values='Revenue',
    index='Month',
    columns='Category',
    aggfunc='mean'
)
pivot_month_mean

Category,Books,Clothing,Electronics,Home,Toys
Month,,,,,
1,700.685000,900.454000,1142.316667,772.211765,1172.391250
2,1272.711667,969.534667,985.441765,1164.355625,1127.982857
3,1023.760000,1015.617333,981.257500,962.896667,1153.322667
